# Fase B: Pipeline ETL - Bodega de Datos de Dengue
**Objetivo:** Extraer datos históricos de SIVIGILA, transformarlos al Modelo Estrella diseñado en la Fase A y cargarlos en una base de datos PostgreSQL.

In [5]:
# Instalación de librerías necesarias (descomentar si falta alguna)
#pip install pandas openpyxl sqlalchemy psycopg2-binary python-dotenv

import pandas as pd
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

# Ocultar advertencias de pandas para una salida más limpia
pd.options.mode.chained_assignment = None

## 1. Extracción

Lectura del Parquet consolidado con pandas y validación de que contiene las columnas que requiere el modelo estrella.

In [ ]:
# Extracción: lectura del Parquet consolidado (salida de procesar.ipynb)
NOMBRE_PARQUET = "dengue_consolidado.parquet"

# Columnas que produce procesar.ipynb y que el modelo de la Fase A necesita
COLUMNAS_ESPERADAS = [
    "ANO", "SEMANA", "COD_DPTO_O", "COD_MUN_O",
    "Departamento_ocurrencia", "Municipio_ocurrencia",
    "AJUSTE", "PAC_HOS", "SEXO", "EDAD", "UNI_MED",
    "AREA", "TIP_SS", "PER_ETN", "GP_DESPLAZ",
    "GP_MIGRANT", "GP_GESTAN", "estrato",
    "TIP_CAS", "nom_est_f_caso",
]

# La ruta se resuelve desde la raíz del proyecto o desde la carpeta entrega/
candidatas = [
    Path("datos/procesados") / NOMBRE_PARQUET,
    Path("../datos/procesados") / NOMBRE_PARQUET,
]
ruta_parquet = next((p for p in candidatas if p.exists()), None)

if ruta_parquet is None:
    raise FileNotFoundError(
        f"No se encontró {NOMBRE_PARQUET}. Rutas buscadas: "
        + ", ".join(str(p.resolve()) for p in candidatas)
        + ". Ejecuta primero procesar.ipynb para generarlo a partir de datos/originales."
    )

try:
    df_consolidado = pd.read_parquet(ruta_parquet)
except (OSError, ValueError, ImportError) as e:
    raise RuntimeError(f"No se pudo leer {ruta_parquet.resolve()}: {e}") from e

# Validaciones de la extracción
if df_consolidado.empty:
    raise ValueError(f"El archivo {ruta_parquet.resolve()} no contiene registros.")

faltantes = [c for c in COLUMNAS_ESPERADAS if c not in df_consolidado.columns]
if faltantes:
    raise ValueError(f"El Parquet no contiene las columnas esperadas: {faltantes}")

print(f"Archivo: {ruta_parquet.resolve()}")
print(f"Registros: {len(df_consolidado):,}")
print(f"Columnas: {len(df_consolidado.columns)}")

# Mostrar una muestra para explorar los datos
display(df_consolidado.head(10))